<a href="https://colab.research.google.com/github/ipreencekmr/anthropic/blob/main/Prompt_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Prompt Evaluation


In [1]:
%pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.8/929.8 kB 4.1 MB/s eta 0:00:00


### Client Setup

In [2]:
#Load Environment Variable
import os
from google.colab import userdata

# Retrieve from Colab Secrets and set to environment variables
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

In [62]:
#Create anthropic client
from anthropic import Anthropic
client = Anthropic()
model = 'claude-haiku-4-5'

### Helper functions

In [63]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[], output_config=None, chat_model=None):
    params = {
        "model": chat_model if chat_model is not None else model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if output_config:
        params["output_config"] = output_config

    response = client.messages.create(**params)
    return response.content[0].text

### Generate Dataset

In [79]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
         "solution_criteria": "Key criteria for evaluating the solution"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [80]:
dataset = generate_dataset()
print(dataset)

[{'task': 'Extract all AWS S3 bucket names from CloudFormation template output text', 'format': 'regex', 'solution_criteria': 'Regex should match S3 bucket naming conventions (lowercase letters, numbers, hyphens) and correctly identify bucket names from various text formats'}, {'task': 'Parse an AWS IAM policy JSON and extract all allowed actions for a specific resource', 'format': 'python', 'solution_criteria': 'Function should correctly parse nested JSON structures, handle multiple statements, and return a list of all actions that apply to the specified resource ARN'}, {'task': 'Create a CloudFormation template JSON structure for an EC2 instance with basic security group and IAM role', 'format': 'json', 'solution_criteria': 'JSON should be valid CloudFormation syntax, include AWSTemplateFormatVersion, Resources section with EC2Instance, SecurityGroup, and IAM role with proper references and dependencies'}]


### Save the dataset

In [81]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

### Code Grader

In [82]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

### Model Grader

In [68]:
grading_schema = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "strengths": {
            "type": "array",
            "items": {"type": "string"},
            "description": "1 to 3 key strengths"
        },
        "weaknesses": {
            "type": "array",
            "items": {"type": "string"},
            "description": "1 to 3 key areas for improvement"
        },
        "reasoning": {
            "type": "string"
        },
        "score": {
            "type": "integer",
            "description": "A whole number from 1 to 10"
        }
    },
    "required": ["strengths", "weaknesses", "reasoning", "score"]
}

In [83]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.

    Original Task:
    <task>
    {test_case["task"]}
    </task>

    Solution to Evaluate:
    <solution>
    {output}
    </solution>

    Criteria you should use to evaluate the solution:
    <criteria>
    {test_case["solution_criteria"]}
    </criteria>

    Output Format
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10

    Respond with JSON. Keep your response concise and direct.
    Example response shape:
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number
    }}
    """

    messages = []
    add_user_message(messages, eval_prompt)

    output_config = {
        "format": {
            "type": "json_schema",
            "schema": grading_schema
        }
    }

    eval_text = chat(messages, chat_model="claude-haiku-4-5-20251001", output_config=output_config)
    try:
        return json.loads(eval_text)
    except json.JSONDecodeError:
        print(f"Failed to parse grading response: {eval_text!r}")
        return {"score": None, "reasoning": "parse error", "strengths": [], "weaknesses": []}

### Run Tests

In [93]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    format_instructions = {
        "json": "Respond with only a single valid JSON object. No explanation, no markdown code fences, no commentary.",
        "python": "Respond with only a single Python function. No explanation, no markdown code fences, no commentary.",
        "regex": "Respond with only a single regular expression pattern. No explanation, no markdown code fences, no commentary, just the pattern itself."
    }

    prompt = f"""
      Please solve the following task:

      {test_case["task"]}

      {format_instructions[test_case["format"]]}
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code") # prefillers
    output = chat(messages, stop_sequences=["```"])
    return output.strip()

In [94]:
def run_test_case(test_case):
    output = run_prompt(test_case)

    # Grade the output
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [95]:
from statistics import mean

def run_eval(dataset):
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [96]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.333333333333334


In [97]:
print(json.dumps(results, indent=2))

[
  {
    "output": "(?:s3:::)?([a-z0-9][a-z0-9\\-]{1,61}[a-z0-9])(?=\\s|,|\"|'|$|/)",
    "test_case": {
      "task": "Extract all AWS S3 bucket names from CloudFormation template output text",
      "format": "regex",
      "solution_criteria": "Regex should match S3 bucket naming conventions (lowercase letters, numbers, hyphens) and correctly identify bucket names from various text formats"
    },
    "score": 8.5,
    "reasoning": "The regex demonstrates solid understanding of S3 constraints and CloudFormation contexts. The 1-61 character range for the middle segment correctly enforces the 3-63 total length requirement. However, the lookahead boundary detection is somewhat restrictive for CloudFormation outputs which may have buckets adjacent to colons (ARNs) or brackets (JSON). The pattern would benefit from testing against real CloudFormation templates to verify coverage."
  },
  {
    "output": "import json\nfrom typing import Set\n\ndef extract_allowed_actions(policy_json: str